# Update trade mixes

This workflow shows how to overwrite the **regional sourcing** of traded items
with `Database.update_trade_mix`, and how to prepare a SUT for agile trade
updates with `Database.pool_trade`.

The trade mix is the dual of the supply mix: instead of redistributing several
labels *within one region*, it redistributes **one item across its origin
regions** inside the columns of each destination market, preserving every
selected column total. Each buyer therefore keeps its total input of the item;
only the sourcing split changes.

Origins that are not listed keep their current share: the provided shares
(summing to one) are rescaled onto the combined share currently held by the
listed origins. Partial updates — e.g. rebalancing only the intra-EU sourcing
while the Chinese share stays put — are therefore well defined.

## Where the trade lives

The rewritten blocks depend on the table layout:

| Layout | Bundle rows | Rewritten columns |
|---|---|---|
| **IOT (Isard)** | `(origin, Sector, item)` | destination's `z` and `Y` columns |
| **SUT, Commodity level (Isard, e.g. raw EXIOBASE)** | `(origin, Commodity, item)` | destination's `u` and `Yc` columns |
| **SUT, Activity level (pooled / Chenery-Moses)** | `(origin, Activity, item)` | destination market columns of `s` selected by `commodities=...` |

## Isard tables

On an Isard table the bilateral sourcing lives in the use side rows. Rewrite
the sourcing of one commodity into one destination market with a
`destination -> {origin: share}` mapping:

In [ ]:
import mario

db = mario.parse_exiobase(
    table="SUT",
    unit="Monetary",
    path="/path/to/MRSUT_2022.zip",
    name="EXIOBASE 3 SUT",
    year=2022,
)

# Italy sources its Motor vehicles 60% domestically and 40% from Germany;
# origins that are not listed (e.g. FR) keep their current share.
db.update_trade_mix(
    {"IT": {"IT": 0.6, "DE": 0.4}},
    items="Motor vehicles, trailers and semi-trailers (34)",
    scenario="baseline",
)

> **Chenery-Moses uniformity — read before using on Isard tables.**
> In an Isard table every buyer of one destination region can have its own
> sourcing profile. Applying one destination-level mix rewrites **all** the
> destination's buyer columns with the *same* origin shares: it imposes the
> Chenery-Moses hypothesis of uniform sourcing on those columns. This matches
> how per-destination statistics (Comtrade, ENTSO-E) are published, and it is
> physically sound for grid-like commodities; for differentiated goods be
> aware that buyer-level sourcing heterogeneity is averaged away. Use
> `column_sectors=...` to leave selected buyers on their original sourcing.

## Pooled tables (`pool_trade`)

`pool_trade` prepares one SUT so that trade updates become market-share
rewrites, decoupled from the technology mixes. For each pooled commodity `c`
it adds, per region, one `"{c} - supply"` pass-through activity and one
`"{c} - need"` market commodity:

1. the pass-through activity consumes the whole domestic output of `c`;
2. every buyer (intermediate use and final demand) moves onto the domestic
   need commodity, keeping its total input;
3. the supply block routes each destination market to the origin pass-through
   activities with the **bilateral trade flows observed on the Isard use
   side** — so the pooled table is economically equivalent to the original at
   the destination level, and the initial market shares in `s` equal the
   observed origin shares.

The technology mix of `c` (on the domestic commodity column) and its trade
mix (on the need column) then live in two separate market-share columns of
`s` and can be updated independently. The pooled pairs are recorded on
`meta.pooled_trade_map`.

> **Notes.** The pooling is structural: it rebuilds the baseline and drops
> other scenarios with one warning. What is averaged away is the
> buyer-specific sourcing heterogeneity inside each destination (the
> Chenery-Moses hypothesis, applied only to the selected commodities). For
> commodities distributed through one shared network — electricity being the
> canonical case — this is typically *more* realistic than the Isard
> buyer-specific sourcing.

In [ ]:
db.pool_trade("Electricity")

# trade mix on the pooled market: shares by origin per destination...
db.update_trade_mix(
    {"IT": {"IT": 0.85, "FR": 0.10, "CH": 0.05}},
    items="Electricity - supply",
    commodities="Electricity - need",
    scenario="baseline",
)

# ...and technology mix on the domestic market, fully decoupled.
db.update_supply_mix(
    {"IT": {"Production of electricity by solar photovoltaic": 0.6,
            "Production of electricity by wind": 0.4}},
    level="Activity",
    commodities="Electricity",
    scenario="baseline",
)

## Excel route

The shock workbook accepts the `Trade mix N` type (with `N` from 1 to 10,
available in the template picklists). All rows sharing the same
`(item, Region_to, 'Trade mix N')` triple describe one destination mix:
`Region_from` carries the origins and `Value` their shares, while `Region_to`
names the destination market — one explicit region, `all` is rejected.

On IOT workbooks author the mix on the `z` (or `Y`) sheet; on SUT workbooks
use the `u`/`Yc` sheets for Isard commodity sourcing and the `s` sheet for
pooled market shares (there `Commodity_to` must name the market commodity,
e.g. `Electricity - need`). For example, on the `s` sheet of one pooled SUT:

| Region_from | Activity_from | Region_to | Commodity_to | type | value |
|---|---|---|---|---|---|
| IT | Electricity - supply | IT | Electricity - need | Trade mix 1 | 0.85 |
| FR | Electricity - supply | IT | Electricity - need | Trade mix 1 | 0.10 |
| CH | Electricity - supply | IT | Electricity - need | Trade mix 1 | 0.05 |

See [Shock analyses](apply_shocks.ipynb) for the general shock workflow.